# **Shapley values with a causal graph: asymmetric, causal Shapley and Shapley Flow**

Practice for the module [“Causality and Linear Models”](https://ai-interpretability.school), lesson “Shapley values with a causal graph”.

In the lesson the variants of Shapley values were compared on two features and one model. Here we compute them with our own code on a bike rental model whose feature graph is known, and check the marginal variant against the `shap` package. We check four claims of the lesson:

1. marginal (interventional) SHAP values give a cause feature nothing if the model does not look at it, while conditional (observational) ones do, but do not distinguish the graph;
2. asymmetric Shapley values (ASV) give the cause all the contribution that passes through its effects;
3. causal Shapley values change together with the graph on the same data;
4. Shapley Flow splits the same prediction over the edges of the graph, and the sum over any cut of the graph is the same.

Plan of the notebook:

1. data and model;
2. four value functions;
3. symmetric and asymmetric Shapley values;
4. a different graph on the same data;
5. Shapley Flow;
6. where it breaks: the instability of ASV;
7. a check against `shap`.

The notebook runs on a CPU, the whole run takes a few seconds.

In [ ]:
import itertools
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

SEED = 0
names = ['season', 'temperature', 'humidity']

## 1. Data and model

Bike rentals depend on temperature and humidity; the season does not act on rentals by itself but sets the temperature. The feature graph is $X_1 \to X_2$, with $X_3$ on its own:

$$X_1 \sim \mathcal N(0,1),\quad X_2 = 0.8\,X_1 + \varepsilon_2,\ \varepsilon_2 \sim \mathcal N(0, 0.6^2),\quad X_3 \sim \mathcal N(0,1),\quad Y = 2X_2 - X_3 + \varepsilon_Y,$$

where $X_1$ is the season (from winter to summer), $X_2$ is the temperature, $X_3$ is the humidity, all standardized; $Y$ is rentals in arbitrary units. The model $f$ is a linear regression on the three features. We explain the prediction for one day: winter ($x_1 = -1$), but warm for winter ($x_2 = 0.5$), humidity a little above normal ($x_3 = 0.3$).

In [ ]:
rng = np.random.default_rng(SEED)
n = 5000
X1 = rng.normal(size=n)
X2 = 0.8 * X1 + rng.normal(0, 0.6, n)
X3 = rng.normal(size=n)
Y = 2 * X2 - X3 + rng.normal(0, 0.5, n)
X = np.c_[X1, X2, X3]

model = LinearRegression().fit(X, Y)
f = lambda z: model.intercept_ + z @ model.coef_
x = np.array([-1.0, 0.5, 0.3])
mu, Sigma = X.mean(0), np.cov(X.T)
print('model coefficients:', dict(zip(names, model.coef_.round(3).tolist())))
print(f'f(x) = {f(x):.3f} | E f = {f(mu):.3f} | f(x) - E f = {f(x) - f(mu):.3f}')

The model barely looks at the season: its coefficient is about zero. The prediction for this day is 0.705 above the mean.

## 2. Four value functions

The Shapley value of feature $i$ is the mean gain of the value function $v(S)$ when $i$ is added to the features already known. The variants differ in what they put in place of the unknown features $\bar S$:

- **marginal** (interventional SHAP in the `shap` package): $v(S) = E\big[f(x_S, X_{\bar S})\big]$, the unknown features from their joint distribution, independently of the known ones;
- **conditional** (observational): $v(S) = E\big[f(X)\mid X_S = x_S\big]$, from the distribution given the known $x_S$;
- **causal** ([Heskes et al., 2020](https://arxiv.org/abs/2011.01625)): $v(S) = E\big[f(X)\mid do(X_S = x_S)\big]$, from the distribution after an intervention along the graph.

The model is linear, so $E[f(\cdot)] = f(E[\cdot])$, and we compute each value function from the mean of the unknown features. The conditional mean for Gaussian features is $\mu_{\bar S} + \Sigma_{\bar S S}\Sigma_{SS}^{-1}(x_S - \mu_S)$. The mean after an intervention is computed by the equations of the graph: a feature from $S$ equals $x_i$, the others follow their equation from the parents already computed. The equation of $X_2$ on $X_1$ is estimated by regression.

In [ ]:
d = 3
eq2 = LinearRegression().fit(X[:, [0]], X[:, 1])        # X2 = c + alpha * X1 + noise
alpha, c2 = eq2.coef_[0], eq2.intercept_
print(f'alpha = {alpha:.3f}')


def v_marginal(S):
    z = mu.copy()
    z[S] = x[S]
    return f(z)


def v_conditional(S):
    z = x.copy()
    R = [j for j in range(d) if j not in S]
    z[R] = mu[R] if not S else mu[R] + Sigma[np.ix_(R, S)] @ np.linalg.solve(Sigma[np.ix_(S, S)], x[S] - mu[S])
    return f(z)


def v_causal_chain(S):
    """Graph X1 -> X2, X3 on its own: features from S are fixed by intervention, the rest follow the equations."""
    z = np.empty(d)
    z[0] = x[0] if 0 in S else mu[0]
    z[1] = x[1] if 1 in S else c2 + alpha * z[0]
    z[2] = x[2] if 2 in S else mu[2]
    return f(z)


for name, v in (('marginal', v_marginal), ('conditional', v_conditional), ('causal', v_causal_chain)):
    print(f'{name:12s}', {str(S): round(float(v(S)), 3) for S in ([], [0], [1], [0, 1])})

The difference is visible already on the coalitions. Knowing the season alone, $S = \{1\}$: the marginal function barely changes, while the conditional and the causal ones fall, since in winter one expects cold. Knowing the temperature alone, $S = \{2\}$: the conditional function pulls the season toward warmth, while the causal one does not, since an intervention on temperature does not change the season.

## 3. Symmetric and asymmetric Shapley values

The Shapley value averages the gain over the orders $\pi$ in which the features are added:

$$\phi_i = \sum_{\pi} w(\pi)\,\big[v(\{j : \pi(j) \le \pi(i)\}) - v(\{j : \pi(j) < \pi(i)\})\big].$$

The symmetric variant takes all $3! = 6$ orders with weight $1/6$. The asymmetric one ([Frye, Rowat, and Feige, 2020](https://arxiv.org/abs/1910.06358)) keeps only the orders where the cause comes before the effect, the season before the temperature: there are three of them, with weight $1/3$.

**Task 1. Write a function `shapley(v, orders)` that averages the gains over a list of orders, and compute the symmetric Shapley values with the causal value function `v_causal_chain`. Give the value for the season, rounded to three decimal places, as the answer.**

In [ ]:
# Your code here

Now all five variants on one prediction.

**Task 2. Compute the asymmetric Shapley values with the conditional value function (these are the ASV of Frye et al.) on the orders where the season comes before the temperature, and put them in a table with the other variants. Give the ASV for the season, rounded to three decimal places, as the answer.**

In [ ]:
# Your code here

The sum is the same in all rows, 0.705, the difference between the prediction and the mean: efficiency holds for all variants. Marginal values give the season almost zero and all the credit to the temperature. ASV give the season the whole effect that goes through the temperature, $\hat\alpha\hat\beta_2(x_1 - \mu_1) \approx -1.6$: in winter there are fewer rentals because it is cold, and this is credited to the season, while the temperature keeps only how much warmer this day is than usual for winter. The symmetric variants split this effect in half between the season and the temperature.

On the chain the conditional and the causal values almost coincide. They differ where the relation between the features does not go from one to the other.

## 4. A different graph on the same data

Suppose the season and the temperature are related not directly but through a common cause, for instance the climate of the region, which is not recorded in the data. The distribution of the features is the same, but an intervention on the season no longer changes the temperature:

$$v(S) = E\big[f(X)\mid do(X_S = x_S)\big],\qquad E[X_2 \mid do(X_1 = x_1)] = \mu_2.$$

In [ ]:
def v_causal_confounded(S):
    """Season and temperature are effects of a common cause: intervening on one does not change the other."""
    z = mu.copy()
    z[S] = x[S]
    return f(z)


compare = pd.DataFrame({
    'conditional (observational SHAP)': shapley(v_conditional, all_orders),
    'ASV, if the season is taken as the cause': shapley(v_conditional, causal_orders),
    'causal, graph with a common cause': shapley(v_causal_confounded, all_orders),
    'causal, chain': shapley(v_causal_chain, all_orders),
}, index=names).T
compare.round(3)

The conditional values and ASV do not depend on the graph: they see only the data distribution, and it is the same. The causal values with the common-cause graph coincide with the marginal ones: the model does not need the season, and it gets almost nothing (0.012). With the chain the season gets its half of the effect through the temperature. Which answer is right, the data cannot say; the graph decides, and its choice has to be written next to the explanation.

## 5. Shapley Flow

[Wang, Wiens, and Lundberg, 2021](https://arxiv.org/abs/2010.14592) distribute the difference $f(x) - f(x')$ over the edges of the graph, where $x'$ is a background point. Every node except the roots has its own noise, which is also a source: $\varepsilon_2 = x_2 - c_2 - \hat\alpha x_1$. The graph for Shapley Flow: $X_1 \to X_2$, $\varepsilon_2 \to X_2$, and the edges $X_1 \to f$, $X_2 \to f$, $X_3 \to f$ into the model.

The Shapley Flow value of a path is the mean gain of the output when the object's value $x$ passes along this path instead of the background value, over all depth-first orders of traversing the graph; an edge gets the sum of the values of the paths through it. When both the model and the equations are linear, the gain of a path does not depend on the order and equals the product of the coefficients along the path times the shift of the source $x_s - x'_s$. The background is the mean point $x' = \mu$, for which $\varepsilon'_2 = 0$.

In [ ]:
b1, b2, b3 = model.coef_
eps2 = x[1] - c2 - alpha * x[0]
paths = {
    'X1 -> f': b1 * (x[0] - mu[0]),
    'X1 -> X2 -> f': alpha * b2 * (x[0] - mu[0]),
    'eps2 -> X2 -> f': b2 * eps2,
    'X3 -> f': b3 * (x[2] - mu[2]),
}
print({p: round(float(val), 3) for p, val in paths.items()})
print(f'sum of paths {sum(paths.values()):.3f} | f(x) - f(mu) {f(x) - f(mu):.3f}')

**Task 3. Compute the Shapley Flow values of the edges $X_1 \to X_2$ and $X_2 \to f$ as the sum of the values of the paths through them. Check that the sum over the cut at the model ($X_1 \to f$, $X_2 \to f$, $X_3 \to f$) and over the cut at the sources ($X_1 \to f$, $X_1 \to X_2$, $\varepsilon_2 \to X_2$, $X_3 \to f$) is the same. Give the value of the edge $\varepsilon_2 \to X_2$, rounded to three decimal places, as the answer.**

In [ ]:
# Your code here

The cut at the model gives the marginal SHAP values: the edge $X_2 \to f$ gets $\hat\beta_2(x_2 - \mu_2)$. The cut at the sources splits the same number deeper: of the temperature's contribution, $-1.6$ came from the season and $+2.6$ from the day being warmer than usual for winter. The edges leaving $X_1$ sum to $-1.595$, the edge $\varepsilon_2 \to X_2$ is $2.592$: these are the causal asymmetric values of the season and the temperature from section 3. The authors prove (section 3.4 of the paper) that with a graph that describes all dependencies of the features, the Shapley Flow values at the sources coincide with ASV; here ASV differ by about 0.013, because the conditional mean is computed from the sample covariance and the equation of the graph by a separate regression.

## 6. Where it breaks: the instability of ASV

[Heskes et al., 2020](https://arxiv.org/abs/2011.01625) show that ASV depend on the order of the features even when there is no relation between them. The model is the XOR of two independent equiprobable binary features; we explain the prediction $f(1, 0) = 1$. Add an edge $X_1 \to X_2$ of zero strength to the graph: the distribution does not change, only the list of admissible orders does.

In [ ]:
xor_point = np.array([1, 0])
grid = np.array(list(itertools.product([0, 1], repeat=2)))     # equiprobable pairs of values


def v_xor(S):
    rows = grid[np.all(grid[:, S] == xor_point[S], axis=1)] if S else grid
    return np.mean(rows[:, 0] ^ rows[:, 1])


def shapley2(v, orders):
    phi = np.zeros(2)
    for order in orders:
        known = []
        for i in order:
            phi[i] += v(known + [i]) - v(known)
            known = known + [i]
    return phi / len(orders)


print('symmetric:', shapley2(v_xor, [(0, 1), (1, 0)]))
print('ASV, X1 before X2:', shapley2(v_xor, [(0, 1)]))

An edge of zero strength moved the whole contribution from $X_1$ to $X_2$: 0.25 and 0.25 became 0 and 0.5. The symmetric values, both conditional and causal, do not change with such an edge. So the asymmetric variant is used when the order of the causes is really known, for instance given by time, not to test a guess about the graph.

## 7. A check against `shap`

Marginal values are computed by `shap` with the `Independent` masker: the unknown features are drawn from a background sample independently of the known ones. The background is all 5000 objects, so the background mean coincides with `mu`.

In [ ]:
import shap

explainer = shap.explainers.Exact(model.predict, shap.maskers.Independent(X, max_samples=len(X)))
phi_shap = explainer(x[None, :]).values[0]
print('shap, Independent:', phi_shap.round(3))
print('our marginal:     ', shapley(v_marginal, all_orders).round(3))

ASV and causal Shapley values are implemented in the R package [`shapr`](https://github.com/NorskRegnesentral/shapr) (MIT license): its function `explain` has the arguments `asymmetric`, `causal_ordering` and `confounding`; the Python wrapper `pyshapr` needs R installed. The authors published the Shapley Flow code without a license. So here we use our own implementation, in which every formula is visible. For a nonlinear model and nonlinear equations the value functions are computed by sampling: the unknown features are sampled from the conditional distribution or from the equations of the graph after the intervention, and $f$ is averaged.

## What to take away

- **The value function decides what is explained.** The marginal one explains the model itself and gives the season almost zero; the conditional and the causal ones explain the prediction together with the world where the season sets the temperature.
- **ASV give the cause everything that goes through its effects.** The season gets −1.6, the temperature gets how much warmer the day is than usual for winter.
- **Causal values depend on the graph, conditional ones do not.** On the same data the graph with a common cause gives the season zero, the chain gives it half of the effect through the temperature.
- **Shapley Flow splits the prediction over edges.** The sum over any cut is the same, the cut at the model gives the marginal values, the cut at the sources gives the causal asymmetric ones.
- **The asymmetric variant is unstable to an edge of zero strength.** On XOR such an edge moves the whole contribution from one feature to the other.

Sources:
1. [Frye, Rowat & Feige, 2020. Asymmetric Shapley values: incorporating causal knowledge into model-agnostic explainability](https://arxiv.org/abs/1910.06358)
2. [Heskes, Sijben, Bucur & Claassen, 2020. Causal Shapley Values: Exploiting Causal Knowledge to Explain Individual Predictions of Complex Models](https://arxiv.org/abs/2011.01625)
3. [Wang, Wiens & Lundberg, 2021. Shapley Flow: A Graph-based Approach to Interpreting Model Predictions](https://arxiv.org/abs/2010.14592)
4. [Janzing, Minorics & Blöbaum, 2020. Feature relevance quantification in explainable AI: A causal problem](https://arxiv.org/abs/1910.13413)